# Working with text data

There are two ways to store text data in pandas:

1. object -dtype NumPy array.

2. StringDtype extension type.

For backwards-compatibility, object dtype remains the default type we infer a list of strings to

In [1]:
import pandas as pd
import numpy as np

In [2]:
pd.Series(["a", "b", "c"])

0    a
1    b
2    c
dtype: object

To explicitly request string dtype, specify the dtype

In [3]:
pd.Series(["a", "b", "c"], dtype="string")

0    a
1    b
2    c
dtype: string

In [4]:
pd.Series(["a", "b", "c"], dtype=pd.StringDtype())

0    a
1    b
2    c
dtype: string

Or astype after the Series or DataFrame is created

In [5]:
s = pd.Series(["a", "b", "c"])
s

0    a
1    b
2    c
dtype: object

In [6]:
s = s.astype("string")
s

0    a
1    b
2    c
dtype: string

## String methods

Series and Index are equipped with a set of string processing methods that make it easy to operate on each element of the array. 

Perhaps most importantly, these methods exclude missing/NA values automatically. These are accessed via the str attribute and generally have names matching the equivalent (scalar) built-in string methods:

In [7]:
s = pd.Series(
    ["A", "B", "C", "Aaba", "Baca", np.nan, "CABA", "dog", "cat"], dtype="string"
)
s

0       A
1       B
2       C
3    Aaba
4    Baca
5    <NA>
6    CABA
7     dog
8     cat
dtype: string

In [8]:
s.str.lower()

0       a
1       b
2       c
3    aaba
4    baca
5    <NA>
6    caba
7     dog
8     cat
dtype: string

In [9]:
s.str.upper()

0       A
1       B
2       C
3    AABA
4    BACA
5    <NA>
6    CABA
7     DOG
8     CAT
dtype: string

In [10]:
s.str.len()

0       1
1       1
2       1
3       4
4       4
5    <NA>
6       4
7       3
8       3
dtype: Int64

In [11]:
idx = pd.Index([" jack", "jill ", " jesse ", "frank"])
idx

Index([' jack', 'jill ', ' jesse ', 'frank'], dtype='object')

hint:  suppose we are looking for a string method that contains the word find somewhere in its name. 

In [12]:
str.*str*?

In [13]:
str.strip?

In [14]:
idx.str.strip()   # try lstrip, rstrip

Index(['jack', 'jill', 'jesse', 'frank'], dtype='object')

The string methods on Index are especially useful for cleaning up or transforming DataFrame columns. For instance, you may have columns with leading or trailing whitespace:

In [15]:
df = pd.DataFrame(
    np.random.randn(3, 2), columns=[" Column A ", " Column B "], index=range(3)
)
df

,Column A,Column B
0,0.929446,0.033319
1,-1.501753,0.038619
2,1.223293,-0.144064


In [16]:
df.columns.str.strip()

Index(['Column A', 'Column B'], dtype='object')

In [17]:
df.columns.str.lower()

Index([' column a ', ' column b '], dtype='object')

These string methods can then be used to clean up the columns as needed. Here we are removing leading and trailing whitespaces, lower casing all names, and replacing any remaining whitespaces with underscores:

In [18]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df

,column_a,column_b
0,0.929446,0.033319
1,-1.501753,0.038619
2,1.223293,-0.144064


## Splitting and replacing strings

In [19]:
s2 = pd.Series(["a_b_c", "c_d_e", np.nan, "f_g_h"], dtype="string")

In [20]:
s2.str.split("_")

0    [a, b, c]
1    [c, d, e]
2         <NA>
3    [f, g, h]
dtype: object

Elements in the split lists can be accessed using get 

In [21]:
s2.str.split("_").str.get(1)

0       b
1       d
2    <NA>
3       g
dtype: object

or [] notation:

In [22]:
s2.str.split("_").str[1]

0       b
1       d
2    <NA>
3       g
dtype: object

In [23]:
str.split?

In [ ]:
pd.Series.str.split?

In [24]:
pd.Series.str.*?

C:\Users\User\anaconda3\lib\site-packages\IPython\utils\wildcard.py:70: FutureWarning: pandas.Float64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  ns[key] = getattr(obj, key)
C:\Users\User\anaconda3\lib\site-packages\IPython\utils\wildcard.py:70: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  ns[key] = getattr(obj, key)
C:\Users\User\anaconda3\lib\site-packages\IPython\utils\wildcard.py:70: FutureWarning: pandas.UInt64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  ns[key] = getattr(obj, key)


In [25]:
s = pd.Series(
    [
        "this is a regular sentence",
        "https://docs.python.org/3/tutorial/index.html",
        np.nan
    ]
)
s

0                       this is a regular sentence
1    https://docs.python.org/3/tutorial/index.html
2                                              NaN
dtype: object

In [26]:
s.str.split()

0                   [this, is, a, regular, sentence]
1    [https://docs.python.org/3/tutorial/index.html]
2                                                NaN
dtype: object

In [30]:
s.str.split(n=2)

0                     [this, is, a regular sentence]
1    [https://docs.python.org/3/tutorial/index.html]
2                                                NaN
dtype: object

In [28]:
s.str.rsplit(n=2)

0                     [this is a, regular, sentence]
1    [https://docs.python.org/3/tutorial/index.html]
2                                                NaN
dtype: object

For slightly more complex use cases like splitting the html document name from a url, a combination of parameter settings can be used.

In [31]:
s.str.rsplit("/", n=1)

0                        [this is a regular sentence]
1    [https://docs.python.org/3/tutorial, index.html]
2                                                 NaN
dtype: object

In [32]:
s.str.rsplit("/")

0                         [this is a regular sentence]
1    [https:, , docs.python.org, 3, tutorial, index...
2                                                  NaN
dtype: object

It is easy to **expand** this to return a DataFrame using expand.

In [34]:
s2

0    a_b_c
1    c_d_e
2     <NA>
3    f_g_h
dtype: string

In [35]:
s2.str.split("_")

0    [a, b, c]
1    [c, d, e]
2         <NA>
3    [f, g, h]
dtype: object

In [33]:
s2.str.split("_", expand=True)

,0,1,2
0,a,b,c
1,c,d,e
2,<NA>,<NA>,<NA>
3,f,g,h


## Extracting substrings

The extract method accepts a regular expression with at least one capture group.

https://docs.python.org/3/library/re.html

In [37]:
pd.Series(
    ["a1", "b2", "c3"],
    dtype="string",
).str.extract(r"([ab])(\d)", expand=True)

,0,1
0,a,1
1,b,2
2,<NA>,<NA>


In [40]:
pd.Series(
    ["a1", "b2", "c3"],
    dtype="string",
).str.extract(r"([ab])(\d)", expand=False)


,0,1
0,a,1
1,b,2
2,<NA>,<NA>


or you can have it optional

In [41]:
pd.Series(
    ["a1", "b2", "3"],
    dtype="string",
).str.extract(r"([ab])?(\d)", expand=False)

,0,1
0,a,1
1,b,2
2,<NA>,3


In [42]:
pd.Series(["a1", "b2", "c3"], dtype="string").str.extract(r"[ab](\d)", expand=True)  # True enforces to generate dataframe


,0
0,1
1,2
2,<NA>


In [43]:
pd.Series(["a1", "b2", "c3"], dtype="string").str.extract(r"[ab](\d)", expand=False)


0       1
1       2
2    <NA>
dtype: string

## Testing for strings that match or contain a pattern

In [44]:
pattern = r"[0-9][a-z]"

In [45]:
pd.Series(
    ["1", "2", "3a", "3b", "03c", "4dx"],
    dtype="string",
).str.contains(pattern)


0    False
1    False
2     True
3     True
4     True
5     True
dtype: boolean

In [46]:
pd.Series(
    ["1", "2", "3a", "3b", "03c", "4dx"],
    dtype="string",
).str.match(pattern)


0    False
1    False
2     True
3     True
4    False
5     True
dtype: boolean

In [47]:
pd.Series(
    ["1", "2", "3a", "3b", "03c", "4dx"],
    dtype="string",
).str.fullmatch(pattern)


0    False
1    False
2     True
3     True
4    False
5    False
dtype: boolean

The distinction between match, fullmatch, and contains is strictness: 
 - fullmatch tests whether the entire string matches the regular expression 
 - match tests whether there is a match of the regular expression that begins at the first character of the string; 
 - contains tests whether there is a match of the regular expression at any position within the string.

## Creating indicator variables

### quick note

what is **dummy variable**? A dummy variable is a binary variable that indicates whether a separate categorical variable takes on a specific value. 

You can extract dummy variables from string columns. For example if they are separated by a '|':

In [48]:
s = pd.Series(["a", "a|b", np.nan, "a|c"], dtype="string")

In [49]:
s

0       a
1     a|b
2    <NA>
3     a|c
dtype: string

In [50]:
s.str.get_dummies(sep="|")

,a,b,c
0,1,0,0
1,1,1,0
2,0,0,0
3,1,0,1


String Index also supports get_dummies which returns a MultiIndex.

In [51]:
idx = pd.Index(["a", "a|b", np.nan, "a|c"])

In [52]:
idx.str.get_dummies(sep="|")

MultiIndex([(1, 0, 0),
            (1, 1, 0),
            (0, 0, 0),
            (1, 0, 1)],
           names=['a', 'b', 'c'])